In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)
if torch.cuda.is_available() else "CPU")

2.10.0+cu128
True
Tesla T4


### Tensor 생성과 기본 연산

In [ ]:
import torch

# 리스트에서 생성
x = torch.tensor([1, 2, 3])
y = torch.zeros(2, 3) # 2x3 영행렬
z = torch.randn(3, 3) # 정규분포 난수

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print(a + b) # tensor([5., 7., 9.])
print(a * b) # tensor([ 4., 10., 18.])
print(a.shape) # torch.Size([3])
print(a.reshape(3, 1)) # 차원 변환


tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
torch.Size([3])
tensor([[1.],
        [2.],
        [3.]])


### Dataset과 DataLoader

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class ImageDataset(Dataset):
  def __init__(self, image_paths, labels, transform=None):
   self.image_paths = image_paths
   self.labels = labels
   self.transform = transform

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    img = Image.open(self.image_paths[idx]).convert("RGB")
    label = self.labels[idx]

    if self.transform:
      img = self.transform(img)

    return img, label


In [ ]:
from torch.utils.data import DataLoader

dataset = ImageDataset(paths, labels, transform)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True # 학습이 끝난 이후에 false
)

for x, y in loader:
  print(x.shape, y.shape)


NameError: name 'paths' is not defined

### Autograd와 자동미분

In [ ]:

# 기울기 추적이 가능한 텐서 생성
x = torch.tensor(2.0, requires_grad=True)

# 순전파 계산
y = x**2 + 3

# 역전파 → dy/dx = 2x = 4.0
y.backward()
print(x.grad) # tensor(4.)


tensor(4.)


### nn.Module로 신경망 구축

In [ ]:
import torch.nn as nn

class SimpleNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(10, 5) # 입력 10 → 은닉 5
    self.fc2 = nn.Linear(5, 1) # 은닉 5 → 출력 1

  def forward(self, x):
    x = torch.relu(self.fc1(x))
    return self.fc2(x)

model = SimpleNet()
input_data = torch.randn(1, 10)
output = model(input_data) # forward() 자동 호출

print(output)


tensor([[0.4775]], grad_fn=<AddmmBackward0>)


### GPU 성능 확인하기

In [ ]:
import torch
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

N = 3000
A = torch.randn(N, N).to(device)
B = torch.randn(N, N).to(device)

start = time.time()
C = A @ B

end = time.time()

print(f"Elapsed time: {end - start:.4f} seconds")


Device: cpu
Elapsed time: 1.6754 seconds


GPU: 0.0003  
CPU: 1.6754

**결론** : GPU가 훨씬 빠름